In [39]:
import getpass
username = getpass.getuser()

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import sys

# Add custom analysis modules
# These contain functions for bandit task analysis and plotting
sys.path.append('/Users/kevinmastro/Library/Mobile Documents/com~apple~CloudDocs/Documents 2/Github/ssm/bandit_models') 
import bandit_plots as bp
import bandit_preprocessing as bpp

sys.path.append('/Users/kevinmastro/Library/Mobile Documents/com~apple~CloudDocs/Documents 2/Github/ssm/bandit_models/bandit_models')
import plot_models_v_mouse as plots
import conditional_probs as cp

# Plotting settings
%matplotlib inline
sns.set(style='ticks', font_scale=1.6, rc={'axes.labelsize':18, 'axes.titlesize':18}) 

# Pandas display options for better exploration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Load the merged data
# data_path = '/Users/kevinmastro/Library/CloudStorage/GoogleDrive-kmastro@broadinstitute.org/Shared drives/Broad Marmoset/2-Experiments/5 - Monkey Logic Task/2 - Raw Data/preprocessed_master_merged_data.csv'
data_path = '/Users/kevinmastro/Library/CloudStorage/GoogleDrive-kmastro@broadinstitute.org/Shared drives/Broad Marmoset/2-Experiments/5 - Monkey Logic Task/2 - Raw Data/Merged_Data/Master_All_Animals_Preprocessed.csv'
df = pd.read_csv(data_path)

# Quick verification
print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Subjects: {df['Animal_Name'].unique() if 'Animal_Name' in df.columns else 'No Subject column found'}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max() if 'Date' in df.columns else 'No Date column'}")

/var/folders/wd/vm48bm495w3czprrgk3t2dq00000gn/T/ipykernel_70730/677851523.py:33: DtypeWarning: Columns (34,38,40,69,70,72,74,83,84,89,90) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


Data loaded successfully!
Shape: (338861, 95)
Subjects: ['Fiona' 'Zazu' 'Sunflower' 'Tuna' 'Raven' 'Knuckles' 'Salmon' 'Griffin'
 'Griffin_Visual_Discrimination_Task_NoInitiation' 'Coral' 'Sonic']
Date range: 2020-12-15 to 2024-03-25


In [40]:
df.columns.unique()

Index(['Trial', 'BlockCount', 'TrialWithinBlock', 'Block', 'Condition',
       'TrialError', 'ReactionTime', 'AbsoluteTrialStartTime', 'Year', 'Month',
       'Day', 'Hour', 'Minute', 'Second', 'BehavioralCodes',
       'ObjectStatusRecord', 'RewardRecord', 'VariableChanges', 'CycleRate_1',
       'CycleRate_2', 'Ver', 'Target1_name', 'Target1_x', 'Target1_y',
       'Target1_xsize', 'Target1_ysize', 'Target1_path', 'Target2_name',
       'Target2_x', 'Target2_y', 'Target2_xsize', 'Target2_ysize',
       'Target2_path', 'RewardVolume', 'RewardType', 'BlockLength',
       'RandomValue', 'Block_Prob', 'StimuliChosen', 'ProbReward', 'Reward',
       'Animal_Name', 'Session_Filename', 'CodeTimes_1', 'CodeTimes_2',
       'CodeTimes_3', 'CodeTimes_4', 'CodeNumbers_1', 'CodeNumbers_2',
       'CodeNumbers_3', 'CodeNumbers_4', 'StartTimes', 'EndTimes',
       'reward_dur', 'EyeOffset_1', 'EyeOffset_2', 'Eye2Offset_1',
       'Eye2Offset_2', 'CodeTimes_5', 'CodeTimes_6', 'CodeTimes_7',
       

In [41]:
# Cell 2: Overview by Animal
print("=== Trials per Animal ===")
print(df['Animal_Name'].value_counts())
print(f"\nTotal sessions: {df['Session_ID'].nunique()}")

# Cell 3: Task Types and Probability Conditions
print("=== Task Types ===")
print(df['Task_Type'].value_counts())

print("\n=== Probability Conditions ===")
print(df['prob_Condition'].value_counts())

print("\n=== Block Probabilities ===")
print(df['Block_Prob'].value_counts())

# Cell 4: Date coverage per animal
print("=== Date Range per Animal ===")
for animal in df['Animal_Name'].dropna().unique():
    animal_data = df[df['Animal_Name'] == animal]
    print(f"{animal}: {animal_data['Date'].min()} to {animal_data['Date'].max()} ({animal_data['Session_ID'].nunique()} sessions)")


=== Trials per Animal ===
Zazu                                               65892
Griffin                                            60048
Raven                                              58239
Fiona                                              48500
Knuckles                                           31317
Sunflower                                          24274
Sonic                                              20212
Salmon                                             17473
Coral                                               6715
Tuna                                                5481
Griffin_Visual_Discrimination_Task_NoInitiation      710
Name: Animal_Name, dtype: int64

Total sessions: 1227
=== Task Types ===
WhereTask              149110
Reversal               104871
WhatTask                40486
Reversal_SetChanges     34533
Shaping                  9861
Name: Task_Type, dtype: int64

=== Probability Conditions ===
80-20      105833
100-0      101555
90-10       52592
100-100 

In [42]:
# Cell: Trials and Sessions by Task Type and Probability Condition

print("=== Trials and Sessions by Task Type ===")
task_summary = df.groupby('Task_Type').agg({
    'Trial': 'count',
    'Session_ID': 'nunique'
}).rename(columns={'Trial': 'Total_Trials', 'Session_ID': 'Num_Sessions'})
print(task_summary)

print("\n=== Trials and Sessions by Probability Condition ===")
prob_summary = df.groupby('prob_Condition').agg({
    'Trial': 'count',
    'Session_ID': 'nunique'
}).rename(columns={'Trial': 'Total_Trials', 'Session_ID': 'Num_Sessions'})
print(prob_summary)

print("\n=== Trials and Sessions by Task Type x Probability Condition ===")
task_prob_summary = df.groupby(['Task_Type', 'prob_Condition']).agg({
    'Trial': 'count',
    'Session_ID': 'nunique'
}).rename(columns={'Trial': 'Total_Trials', 'Session_ID': 'Num_Sessions'})
print(task_prob_summary)

print("\n=== Average Trials per Session by Task Type ===")
trials_per_session = df.groupby(['Session_ID', 'Task_Type']).size().reset_index(name='Trials')
avg_trials = trials_per_session.groupby('Task_Type')['Trials'].agg(['mean', 'std', 'min', 'max'])
print(avg_trials)

=== Trials and Sessions by Task Type ===
                     Total_Trials  Num_Sessions
Task_Type                                      
Reversal                   104871           386
Reversal_SetChanges         34533           116
Shaping                      9861            12
WhatTask                    40486           108
WhereTask                  149110           605

=== Trials and Sessions by Probability Condition ===
                Total_Trials  Num_Sessions
prob_Condition                            
0-0                        2             2
100-0                 101555           462
100-100                 9944            23
80-20                 105833           301
80-80                     11             1
90-10                  52592           236
90-90                      4             3

=== Trials and Sessions by Task Type x Probability Condition ===
                                    Total_Trials  Num_Sessions
Task_Type           prob_Condition                   

In [43]:
df.columns.unique()

Index(['Trial', 'BlockCount', 'TrialWithinBlock', 'Block', 'Condition',
       'TrialError', 'ReactionTime', 'AbsoluteTrialStartTime', 'Year', 'Month',
       'Day', 'Hour', 'Minute', 'Second', 'BehavioralCodes',
       'ObjectStatusRecord', 'RewardRecord', 'VariableChanges', 'CycleRate_1',
       'CycleRate_2', 'Ver', 'Target1_name', 'Target1_x', 'Target1_y',
       'Target1_xsize', 'Target1_ysize', 'Target1_path', 'Target2_name',
       'Target2_x', 'Target2_y', 'Target2_xsize', 'Target2_ysize',
       'Target2_path', 'RewardVolume', 'RewardType', 'BlockLength',
       'RandomValue', 'Block_Prob', 'StimuliChosen', 'ProbReward', 'Reward',
       'Animal_Name', 'Session_Filename', 'CodeTimes_1', 'CodeTimes_2',
       'CodeTimes_3', 'CodeTimes_4', 'CodeNumbers_1', 'CodeNumbers_2',
       'CodeNumbers_3', 'CodeNumbers_4', 'StartTimes', 'EndTimes',
       'reward_dur', 'EyeOffset_1', 'EyeOffset_2', 'Eye2Offset_1',
       'Eye2Offset_2', 'CodeTimes_5', 'CodeTimes_6', 'CodeTimes_7',
       

In [44]:
for filename in df.filename.unique():
    print(filename)

AttributeError: 'DataFrame' object has no attribute 'filename'

In [45]:
# Cell: Task Descriptions from Condition Files

print("=" * 80)
print("TASK DESCRIPTIONS")
print("=" * 80)

print("\n1. WHAT TASK (Stimuli-Based Discrimination)")
print("   Description: Animals choose between visual stimuli based on which stimulus")
print("               is rewarded (not location)")
print("   Probability: 100-0 only (deterministic)")
print("   Files: 'Visual_Discrimination_Task' or just animal name + date")
print("   Notes: Always uses 'On Error = Repeat Immediately'")

print("\n2. WHERE TASK (Location-Based Choice)")
print("   Description: Animals choose based on screen location (left vs right)")
print("   Probability Phases:")
print("      - Phase 1 (WhereBlocks1.X): 100-0 probability")
print("      - Phase 2 (WhereBlocks2.X): 90-10 probability") 
print("      - Phase 3 (WhereBlocks3.X): 80-20 probability")
print("      - Phase 4 (WhereBlocks4.X): Mixed probabilities (rare)")
print("   Block Length: Phase 1 starts at 50 trials, decreases to 20; Phases 2-4 use 20 trials")
print("   Notes: Uses 'On Error = Ignore'")

print("\n3. REVERSAL TASK (Location Reversal Learning)")
print("   Description: Location-based choice where rewarded location flips each block")
print("   Probability Phases:")
print("      - Phase 1 (Reversal1.X): 100-0 probability, 20 trials/block")
print("      - Phase 2 (Reversal2.X): 90-10 probability, 20 trials/block")
print("      - Phase 3 (Reversal3.X): 80-20 probability, 20 trials/block")
print("   Block randomization: Random without replacement")
print("   Notes: Uses 'On Error = Ignore'; stimuli identical, only location differs")

print("\n" + "=" * 80)
print("FILENAME STRUCTURE: YYMMDD AnimalName TaskPhase.Version.csv")
print("Examples:")
print("  - 20230627 Sonic WhereBlocks1.5.csv → Where Task Phase 1 (100-0)")
print("  - 20231102 Sonic Reversal3.1.csv → Reversal Phase 3 (80-20)")
print("=" * 80)

TASK DESCRIPTIONS

1. WHAT TASK (Stimuli-Based Discrimination)
   Description: Animals choose between visual stimuli based on which stimulus
               is rewarded (not location)
   Probability: 100-0 only (deterministic)
   Files: 'Visual_Discrimination_Task' or just animal name + date
   Notes: Always uses 'On Error = Repeat Immediately'

2. WHERE TASK (Location-Based Choice)
   Description: Animals choose based on screen location (left vs right)
   Probability Phases:
      - Phase 1 (WhereBlocks1.X): 100-0 probability
      - Phase 2 (WhereBlocks2.X): 90-10 probability
      - Phase 3 (WhereBlocks3.X): 80-20 probability
      - Phase 4 (WhereBlocks4.X): Mixed probabilities (rare)
   Block Length: Phase 1 starts at 50 trials, decreases to 20; Phases 2-4 use 20 trials
   Notes: Uses 'On Error = Ignore'

3. REVERSAL TASK (Location Reversal Learning)
   Description: Location-based choice where rewarded location flips each block
   Probability Phases:
      - Phase 1 (Reversal1.X): 

In [46]:
# Cell: Block Analysis by Session and Task Phase

print("=== Blocks per Session Analysis ===\n")

# Create a summary of blocks for each session
session_blocks = df.groupby(['Session_ID', 'Task_Type', 'prob_Condition', 'Animal_Name', 'Date']).agg({
    'Block': ['nunique', lambda x: sorted(x.unique())]
}).reset_index()

session_blocks.columns = ['Session_ID', 'Task_Type', 'prob_Condition', 'Animal_Name', 'Date', 'Num_Blocks', 'Block_IDs']

# Summary statistics
print("Block Count Distribution:")
print(session_blocks['Num_Blocks'].value_counts().sort_index())

print("\n=== Block Patterns by Task Type and Probability ===")
for task in df['Task_Type'].dropna().unique():
    print(f"\n{task}:")
    task_sessions = session_blocks[session_blocks['Task_Type'] == task]
    
    for prob in sorted(task_sessions['prob_Condition'].dropna().unique()):
        prob_sessions = task_sessions[task_sessions['prob_Condition'] == prob]
        print(f"\n  {prob} probability:")
        print(f"    Total sessions: {len(prob_sessions)}")
        
        # Count unique block patterns
        block_patterns = prob_sessions['Block_IDs'].apply(lambda x: str(x)).value_counts()
        print(f"    Unique block patterns: {len(block_patterns)}")
        print(f"    Most common patterns:")
        for pattern, count in block_patterns.head(5).items():
            print(f"      {pattern}: {count} sessions")

=== Blocks per Session Analysis ===

Block Count Distribution:
1      94
2     211
3      44
4     264
5      34
6      55
7      23
8      30
9      26
10     32
11     13
12    202
Name: Num_Blocks, dtype: int64

=== Block Patterns by Task Type and Probability ===

WhereTask:

  100-0 probability:
    Total sessions: 363
    Unique block patterns: 77
    Most common patterns:
      [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]: 82 sessions
      [1, 2, 3, 4, 5, 6]: 19 sessions
      [1, 2, 3, 4]: 19 sessions
      [1]: 17 sessions
      [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]: 17 sessions

  100-100 probability:
    Total sessions: 8
    Unique block patterns: 2
    Most common patterns:
      [1]: 7 sessions
      [2]: 1 sessions

  80-20 probability:
    Total sessions: 107
    Unique block patterns: 39
    Most common patterns:
      [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]: 67 sessions
      [1, 2, 5, 6, 8, 12]: 2 sessions
      [1, 6, 9, 11]: 2 sessions
      [2, 4, 7, 10]: 1 sessions
      [5

In [47]:
# Cell: Reversal Task Structure - Reversals vs Cued Transitions

print("=" * 80)
print("REVERSAL TASK STRUCTURE")
print("=" * 80)

print("\n**WITHIN-PAIR TRANSITIONS = TRUE REVERSALS**")
print("  1 → 2: REVERSAL (same stimuli, reward contingencies flip)")
print("  3 → 4: REVERSAL (same stimuli, reward contingencies flip)")
print("  5 → 6: REVERSAL (same stimuli, reward contingencies flip)")
print("  7 → 8: REVERSAL (same stimuli, reward contingencies flip)")

print("\n**BETWEEN-PAIR TRANSITIONS = CUED BLOCK TRANSITIONS**")
print("  2 → 3: CUED TRANSITION (stimuli locations change)")
print("  4 → 5: CUED TRANSITION (stimuli locations change)")
print("  6 → 7: CUED TRANSITION (stimuli locations change)")

print("\n" + "=" * 80)
print("SESSION TYPES:")
print("=" * 80)

print("\n1. TWO-BLOCK SESSIONS [1, 2] - '2-ABT' (2-Alternative Bandit Task)")
print("   • 1 pure reversal")
print("   • 0 cued transitions")
print("   • Stimuli stay IDENTICAL throughout entire session")
print("   • Tests pure reversal learning without location cues")
print(f"   • Count: {len(session_blocks[(session_blocks['Task_Type']=='Reversal') & (session_blocks['Block_IDs'].apply(lambda x: x==[1,2]))])} sessions")

print("\n2. FOUR-BLOCK SESSIONS [1, 2, 3, 4]")
print("   • 2 reversals (1→2, 3→4)")
print("   • 1 cued transition (2→3)")
print("   • Tests both reversal learning AND ability to handle location changes")
print(f"   • Count: {len(session_blocks[(session_blocks['Task_Type']=='Reversal') & (session_blocks['Block_IDs'].apply(lambda x: x==[1,2,3,4]))])} sessions")

print("\n3. EXTENDED SESSIONS [1-8], [5-8], etc.")
print("   • Multiple reversals and cued transitions")
print("   • Same pattern repeats: pairs for reversals, transitions between pairs")

REVERSAL TASK STRUCTURE

**WITHIN-PAIR TRANSITIONS = TRUE REVERSALS**
  1 → 2: REVERSAL (same stimuli, reward contingencies flip)
  3 → 4: REVERSAL (same stimuli, reward contingencies flip)
  5 → 6: REVERSAL (same stimuli, reward contingencies flip)
  7 → 8: REVERSAL (same stimuli, reward contingencies flip)

**BETWEEN-PAIR TRANSITIONS = CUED BLOCK TRANSITIONS**
  2 → 3: CUED TRANSITION (stimuli locations change)
  4 → 5: CUED TRANSITION (stimuli locations change)
  6 → 7: CUED TRANSITION (stimuli locations change)

SESSION TYPES:

1. TWO-BLOCK SESSIONS [1, 2] - '2-ABT' (2-Alternative Bandit Task)
   • 1 pure reversal
   • 0 cued transitions
   • Stimuli stay IDENTICAL throughout entire session
   • Tests pure reversal learning without location cues
   • Count: 143 sessions

2. FOUR-BLOCK SESSIONS [1, 2, 3, 4]
   • 2 reversals (1→2, 3→4)
   • 1 cued transition (2→3)
   • Tests both reversal learning AND ability to handle location changes
   • Count: 157 sessions

3. EXTENDED SESSIONS [

In [48]:
# Cell: Count Reversals and Cued Transitions per Session

def count_reversals_and_transitions(block_ids):
    """Count reversals (within pairs) and cued transitions (between pairs)"""
    reversals = 0
    cued_transitions = 0
    
    for i in range(len(block_ids) - 1):
        current = block_ids[i]
        next_block = block_ids[i + 1]
        
        # Check if consecutive blocks
        if next_block == current + 1:
            # Check if within a pair (odd→even: 1→2, 3→4, 5→6, 7→8)
            if current % 2 == 1:  # Odd block number
                reversals += 1
            # Between pairs (even→odd: 2→3, 4→5, 6→7)
            else:  # Even block number
                cued_transitions += 1
    
    return reversals, cued_transitions

# Apply to Reversal sessions only
reversal_sessions = session_blocks[session_blocks['Task_Type'] == 'Reversal'].copy()
reversal_sessions['Reversals'], reversal_sessions['Cued_Transitions'] = zip(
    *reversal_sessions['Block_IDs'].apply(count_reversals_and_transitions)
)

print("=== Reversal Task: Reversals vs Cued Transitions ===\n")

print("Sessions by type:")
print(f"Pure 2-ABT (1 reversal, 0 transitions): {len(reversal_sessions[(reversal_sessions['Reversals']==1) & (reversal_sessions['Cued_Transitions']==0)])}")
print(f"4-block (2 reversals, 1 transition): {len(reversal_sessions[(reversal_sessions['Reversals']==2) & (reversal_sessions['Cued_Transitions']==1)])}")
print(f"6-block (3 reversals, 2 transitions): {len(reversal_sessions[(reversal_sessions['Reversals']==3) & (reversal_sessions['Cued_Transitions']==2)])}")
print(f"8-block (4 reversals, 3 transitions): {len(reversal_sessions[(reversal_sessions['Reversals']==4) & (reversal_sessions['Cued_Transitions']==3)])}")

print("\n=== Distribution by Probability Condition ===")
for prob in sorted(reversal_sessions['prob_Condition'].dropna().unique()):
    prob_data = reversal_sessions[reversal_sessions['prob_Condition'] == prob]
    print(f"\n{prob}:")
    print(f"  2-ABT (pure reversal): {len(prob_data[(prob_data['Reversals']==1) & (prob_data['Cued_Transitions']==0)])}")
    print(f"  4-block (mixed): {len(prob_data[(prob_data['Reversals']==2) & (prob_data['Cued_Transitions']==1)])}")

=== Reversal Task: Reversals vs Cued Transitions ===

Sessions by type:
Pure 2-ABT (1 reversal, 0 transitions): 169
4-block (2 reversals, 1 transition): 176
6-block (3 reversals, 2 transitions): 0
8-block (4 reversals, 3 transitions): 0

=== Distribution by Probability Condition ===

0-0:
  2-ABT (pure reversal): 0
  4-block (mixed): 0

100-0:
  2-ABT (pure reversal): 20
  4-block (mixed): 49

100-100:
  2-ABT (pure reversal): 0
  4-block (mixed): 0

80-20:
  2-ABT (pure reversal): 100
  4-block (mixed): 81

80-80:
  2-ABT (pure reversal): 0
  4-block (mixed): 0

90-10:
  2-ABT (pure reversal): 49
  4-block (mixed): 46


In [49]:
# SAVE UPDATED MASTER FILE
OUTPUT_DIR = '/Users/kevinmastro/Library/CloudStorage/GoogleDrive-kmastro@broadinstitute.org/Shared drives/Broad Marmoset/2-Experiments/5 - Monkey Logic Task/2 - Raw Data/Merged_Data/'
master_output_file = os.path.join(OUTPUT_DIR, 'Master_All_Animals_Preprocessed_updated.csv')
df.to_csv(master_output_file, index=False)
print("✓ Master file updated")

✓ Master file updated


In [38]:
df

,Trial,BlockCount,TrialWithinBlock,Block,Condition,TrialError,ReactionTime,AbsoluteTrialStartTime,Year,Month,Day,Hour,Minute,Second,BehavioralCodes,ObjectStatusRecord,RewardRecord,VariableChanges,CycleRate_1,CycleRate_2,Ver,Target1_name,Target1_x,Target1_y,Target1_xsize,Target1_ysize,Target1_path,Target2_name,Target2_x,Target2_y,Target2_xsize,Target2_ysize,Target2_path,RewardVolume,RewardType,BlockLength,RandomValue,Block_Prob,StimuliChosen,ProbReward,Reward,Animal_Name,Session_Filename,CodeTimes_1,CodeTimes_2,CodeTimes_3,CodeTimes_4,CodeNumbers_1,CodeNumbers_2,CodeNumbers_3,CodeNumbers_4,StartTimes,EndTimes,reward_dur,EyeOffset_1,EyeOffset_2,Eye2Offset_1,Eye2Offset_2,CodeTimes_5,CodeTimes_6,CodeTimes_7,CodeNumbers_5,CodeNumbers_6,CodeNumbers_7,Date,Outcome,Target1_location,Target2_location,Session_ID,PhysicalChoice,PrevPhysicalChoice,PhysicalSwitch,TaskObject1_Type,TaskObject1_Size_1,TaskObject1_Size_2,TaskObject1_Color_1,TaskObject1_Color_2,TaskObject1_Color_3,TaskObject1_Fill,TaskObject1_LocX,TaskObject1_LocY,TaskObject2_Type,TaskObject2_Source,TaskObject2_LocX,TaskObject2_LocY,TaskObject2_Size1,TaskObject2_Size2,TaskObject3_Type,TaskObject3_Source,TaskObject3_LocX,TaskObject3_LocY,TaskObject3_Size1,TaskObject3_Size2,prob_Condition,Task_Type
0,1,1,1,1,1,6,NaN,0.000000e+00,2021,10,15,12,28,40.427,NaN,NaN,NaN,NaN,49.9975,32.9957,4,pic,7.0,12.0,300.0,300.0,C:\Users\marmo\Documents\MATLAB\MarmosetTask_L...,pic,-7.0,-2.0,300.0,300.0,C:\Users\marmo\Documents\MATLAB\MarmosetTask_L...,0.08,NaN,9999.0,NaN,NaN,TargetB,0.0,Unrewarded,Fiona,20211015_Fiona_100%Con_WhereBlocks1.1_Easy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-10-15,0,TopRight,BottomLeft,2021-10-15_20211015_Fiona_100%Con_WhereBlocks1...,BottomLeft,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100-0,WhereTask
1,2,1,2,1,1,6,NaN,6.885500e+04,2021,10,15,12,29,49.282,NaN,NaN,NaN,NaN,33.3844,11.3957,4,pic,7.0,12.0,300.0,300.0,C:\Users\marmo\Documents\MATLAB\MarmosetTask_L...,pic,-7.0,-2.0,300.0,300.0,C:\Users\marmo\Documents\MATLAB\MarmosetTask_L...,0.08,NaN,9999.0,NaN,NaN,TargetB,0.0,Unrewarded,Fiona,20211015_Fiona_100%Con_WhereBlocks1.1_Easy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-10-15,0,TopRight,BottomLeft,2021-10-15_20211015_Fiona_100%Con_WhereBlocks1...,BottomLeft,BottomLeft,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100-0,WhereTask
2,3,1,3,1,1,6,NaN,8.553550e+04,2021,10,15,12,30,5.962,NaN,NaN,NaN,NaN,33.3318,7.7382,4,pic,7.0,12.0,300.0,300.0,C:\Users\marmo\Documents\MATLAB\MarmosetTask_L...,pic,-7.0,-2.0,300.0,300.0,C:\Users\marmo\Documents\MATLAB\MarmosetTask_L...,0.08,NaN,9999.0,NaN,NaN,TargetB,0.0,Unrewarded,Fiona,20211015_Fiona_100%Con_WhereBlocks1.1_Easy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-10-15,0,TopRight,BottomLeft,2021-10-15_20211015_Fiona_100%Con_WhereBlocks1...,BottomLeft,BottomLeft,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100-0,WhereTask
3,4,1,4,1,1,0,NaN,9.276853e+04,2021,10,15,12,30,13.195,NaN,NaN,NaN,NaN,33.3414,5.9564,4,pic,7.0,12.0,300.0,300.0,C:\Users\marmo\Documents\MATLAB\MarmosetTask_L...,pic,-7.0,-2.0,300.0,300.0,C:\Users\marmo\Documents\MATLAB\MarmosetTask_L...,0.08,NaN,9999.0,NaN,NaN,TargetA,1.0,Rewarded,Fiona,20211015_Fiona_100%Con_WhereBlocks1.1_Easy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-10-15,1,TopRight,BottomLeft,2021-10-15_20211015_Fiona_100%Con_WhereBlocks1...,TopRight,BottomLeft,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100-0,WhereTask
4,5,1,5,1,1,6,NaN,1.082510e+05,2021,10,15,12,30,28.677,NaN,NaN,NaN,NaN,33.3410,17.9417,4,pic,7.0,12.0,300.0,300.0,C:\Users\marmo\Documents\MATLAB\MarmosetTask_L...,pic,-7.0,-2.0,300.0,300.0,C:\Users\marmo\Documents\MATLAB\MarmosetTas